## Evaluation of poetry_analysis on test sets

Sjekk dikt-utforskning om jeg har gjort denne evalueringen der 
- [x] Hent testsettene som egne filer inn i "tests"-mappen
- [x] Kjør evalueringen som "integrasjonstester"
- [x] Print oppsummeringsrapport m/ precision, recall, F1-score 

## End rhymes



### Multilabel classification

For multilabel classification algorithms, we compute accuracy as number of correct labels divided by number of wrong classifications. The label is called the "rhyme_tag" in this testset. 

Evaluate overall accuracy of rhyme tag (multilabel classification) on:
- Orthographic texts
- Phonemic transcriptions

In [ ]:
import pandas as pd

rhyme_testset = pd.read_excel("evaluation_data/norn_dikt_testsett_rhyme.xlsx")

In [3]:
correct_text = (rhyme_testset.rhyme_tag_text == rhyme_testset.rhyme_tag_gold).sum()
correct_syll = (rhyme_testset.rhyme_tag_syll == rhyme_testset.rhyme_tag_gold).sum()
all_labels = rhyme_testset.rhyme_tag_gold.count()

### Overall accuracy of end rhyme tags in orthographic texts 

In [4]:
accuracy_text = correct_text / all_labels

print(f"the rhyme tagger annotates end rhymes on orthographic text with an accuracy of {accuracy_text * 100:.2f}%")

the rhyme tagger annotates end rhymes on orthographic text with an accuracy of 78.38%


### Overall accuracy of end rhyme tags in phonemic transcriptions 

The transcriptions have been predicted with a G2P-model for modern Norwegian bokmål and the eastern Norwegian dialect.

In [5]:
accuracy_syll = correct_syll / all_labels

print(
    f"the rhyme tagger annotates end rhymes on phonemic transcriptions with an accuracy of {accuracy_syll * 100:.2f}%"
)

the rhyme tagger annotates end rhymes on phonemic transcriptions with an accuracy of 68.38%


### Binary detection of rhyme

When we switch from evaluating whether the algorithm got the rhyme tags correct, to evaluating whether it has correctly detected that a line rhymes with another line, the accuracy increases, as expected. The labels are incrementally assigned starting from the beginning of the alphabet, so the pattern is skewed if two lines are mislabelled early in a long stanza.


In [10]:
correct_text_score = (rhyme_testset["rhyme_score_text"] == rhyme_testset["rhyme_score_gold"]).sum()
correct_syll_score = (rhyme_testset["rhyme_score_syll"] == rhyme_testset["rhyme_score_gold"]).sum()
all_scores = rhyme_testset["rhyme_score_gold"].count()

### Overall accuracy of end rhyme detection in orthographic texts 

In [8]:
accuracy_text = correct_text_score / all_scores

print(
    f"the rhyme tagger annotates (binary) end rhymes on orthographic text with an accuracy of {accuracy_text * 100:.2f}%"
)

the rhyme tagger annotates (binary) end rhymes on orthographic text with an accuracy of 85.42%


### Overall accuracy of end rhyme detection in phonemic transcriptions

In [11]:
accuracy_syll = correct_syll_score / all_scores

print(
    f"the rhyme tagger annotates (binary) end rhymes on phonemic transcriptions with an accuracy of {accuracy_syll * 100:.2f}%"
)

the rhyme tagger annotates (binary) end rhymes on phonemic transcriptions with an accuracy of 80.69%


## Anaphora

Evaluate precision / recall on the following categories of anaphoric patterns:

- [Line](#line-anaphora): Line-initial word(s) repeated within same line
- [Stanza](#stanza-anaphora): line-initial word(s) repeated within same stanza
- Across: Stanza-initial line or partial line repeated across stanzas 
 

In [92]:
import pandas as pd

anaphora_testset = pd.read_excel("evaluation_data/norn_dikt_testsett_anafor.xlsx")

anaphora_testset["anaphora_gold"] = anaphora_testset.anaphora_gold.str.strip()

### Line anaphora

In [93]:
from poetry_analysis.anaphora import extract_line_anaphora

# Get predictions of repeated line-initial phrases
anaphora_testset["anaphora_line_prediction"] = anaphora_testset.text.apply(
    lambda x: extract_line_anaphora(x).get("phrase", None)
)

# Count true positives, true negatives, false positives and false negatives
true_positive = (
    (anaphora_testset.anaphora_line == "x")
    & (anaphora_testset.anaphora_gold == anaphora_testset.anaphora_line_prediction)
).sum()

false_positive = (
    anaphora_testset.anaphora_line_prediction.notna()
    & (anaphora_testset.anaphora_gold != anaphora_testset.anaphora_line_prediction)
).sum()

true_negative = (anaphora_testset.anaphora_gold.isna() & anaphora_testset.anaphora_line_prediction.isna()).sum()

false_negative = ((anaphora_testset.anaphora_line == "x") & (anaphora_testset.anaphora_line_prediction.isna())).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")

print(f"F1 score: {f1_score * 100:2f}%")

Precision: 5.821918%
Recall: 100.000000%
F1 score: 11.003236%


### Stanza anaphora

In [107]:
## Annotate stanza anaphoras with poetry_analysis
from poetry_analysis.anaphora import extract_anaphora

# group lines in the testset into stanzas
anaphora_testset[["poem_id", "stanza_id", "versenumber"]] = anaphora_testset["verse_id"].str.split("_", expand=True)

grouped_texts = anaphora_testset.groupby(["poem_id", "stanza_id"])["text"].agg(list)

# Extract line-initial phrases that are repeated within a stanza
results = grouped_texts.apply(extract_anaphora)

# Flatten `results` into a lookup, on the string verse_id -> overlap
pred_map = {}
for (poem_id, stanza_id), stanza_dict in results.items():  # type: ignore
    if not isinstance(stanza_dict, dict):
        continue
    for verse_num, payload in stanza_dict.items():
        overlap = payload.get("overlap") if isinstance(payload, dict) else None
        pred_map[f"{poem_id}_{stanza_id}_v{verse_num}"] = overlap

# Map predictions back to each original row
anaphora_testset["anaphora_stanza_prediction"] = anaphora_testset.verse_id.apply(lambda v: pred_map.get(v))

anaphora_testset = anaphora_testset.drop(["poem_id", "stanza_id", "versenumber"], axis=1)
# Optional: quick check
anaphora_testset[["verse_id", "text", "anaphora_gold", "anaphora_stanza_prediction"]].head()

,verse_id,text,anaphora_gold,anaphora_stanza_prediction
0,p20_s0_v0,Tvende floder flyder om helvedes hegn.,NaN,None
1,p20_s0_v1,Den ene hvirvler en sydende strom,den,None
2,p20_s0_v2,med glans af smeltet metal.,NaN,None
3,p20_s0_v3,"Den anden vælter sorte,",den,None
4,p20_s0_v4,iskolde vande.,NaN,None


In [108]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (
    (anaphora_testset.anaphora_stanza == "x")
    & (anaphora_testset.anaphora_gold == anaphora_testset.anaphora_stanza_prediction)
).sum()

false_positive = (
    anaphora_testset.anaphora_stanza_prediction.notna()
    & (anaphora_testset.anaphora_gold != anaphora_testset.anaphora_stanza_prediction)
).sum()

true_negative = (anaphora_testset.anaphora_gold.isna() & anaphora_testset.anaphora_stanza_prediction.isna()).sum()

false_negative = (
    (anaphora_testset.anaphora_stanza == "x") & (anaphora_testset.anaphora_stanza_prediction.isna())
).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")

print(f"F1 score: {f1_score * 100:2f}%")

Precision: 50.588235%
Recall: 34.262948%
F1 score: 40.855107%


In [109]:
## Eyeballing errors
condition = (anaphora_testset.anaphora_stanza == "x") & (anaphora_testset.anaphora_stanza_prediction.isna())

anaphora_testset[condition]

,filename,book,verse_id,language,text,anaphora_gold,anaphora_stanza,anaphora_line,anaphora_across,anaphora_prediction,anaphora_line_prediction,anaphora_stanza_prediction,anaphora_across_prediction
1,20_Vision_no-nb_digibok_2009032303011,2009032303011,p20_s0_v1,rm,Den ene hvirvler en sydende strom,den,x,NaN,NaN,NaN,den,None,None
3,20_Vision_no-nb_digibok_2009032303011,2009032303011,p20_s0_v3,rm,"Den anden vælter sorte,",den,x,NaN,NaN,NaN,den,None,None
5,20_Vision_no-nb_digibok_2009032303011,2009032303011,p20_s0_v5,rm,"Fra den ene stiger en varm, rødlig damp,",fra den,x,NaN,NaN,NaN,None,None,None
9,20_Vision_no-nb_digibok_2009032303011,2009032303011,p20_s1_v0,rm,"Den ene med deres, hvis lidenskab",den,x,NaN,NaN,NaN,den,None,None
11,20_Vision_no-nb_digibok_2009032303011,2009032303011,p20_s1_v2,rm,den anden med de dode,den,x,NaN,NaN,NaN,den,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3046,1240_2010020803026.txt,2010020803026,p1240_s5_v1,Landsmål,jeg gik mig ud i lund.,jeg,x,NaN,NaN,NaN,None,None,None
3097,3347_2012030706060.txt,2012030706060,p3347_s0_v0,rm,"Om jeg tilgiver? Spørg mig, om jeg elsker,",om,x,NaN,NaN,NaN,om jeg,None,None
3098,3347_2012030706060.txt,2012030706060,p3347_s0_v1,rm,"Og om jeg ikke selv en synder er,",og om,x,NaN,NaN,NaN,None,None,None
3100,3347_2012030706060.txt,2012030706060,p3347_s0_v3,rm,"Om ei jeg ser, hvor dyb din anger er.",om,x,NaN,NaN,NaN,None,None,None


### Anaphora across stanzas

In [115]:
## Annotate stanza initial lines repeating across stanzas with poetry_analysis
from poetry_analysis.anaphora import extract_anaphora

# group lines in the testset into lists of stanzas
stanza_texts = (
    anaphora_testset.assign(
        poem_id=anaphora_testset["verse_id"].str.extract(r"p(\d+)").astype(int),
        _stanza_num=anaphora_testset["verse_id"].str.extract(r"s(\d+)").astype(int),
        _verse_num=anaphora_testset["verse_id"].str.extract(r"v(\d+)").astype(int),
    )
    .sort_values(["poem_id", "_stanza_num", "_verse_num"])
    .groupby(["poem_id", "_stanza_num"], sort=False)["text"]
    .apply("\n".join)
)

grouped_texts = stanza_texts.groupby(level=0, sort=False).apply(list)

# extract stanza-initial lines that are repeated in several stanzas
results = grouped_texts.apply(extract_anaphora)


# Flatten `results` into a lookup, on the string verse_id -> overlap
pred_map = {}
for poem_id, stanza_dict in results.items():  # type: ignore
    if not isinstance(stanza_dict, dict):
        continue
    for stanza_num, payload in stanza_dict.items():
        overlap = payload.get("overlap") if isinstance(payload, dict) else None
        pred_map[f"p{poem_id}_s{stanza_num}_v0"] = overlap

# Map predictions back to each original row
anaphora_testset["anaphora_across_prediction"] = anaphora_testset.verse_id.apply(lambda v: pred_map.get(v))

# Optional: quick check
anaphora_testset[["verse_id", "text", "anaphora_gold", "anaphora_across_prediction"]].head()

,verse_id,text,anaphora_gold,anaphora_across_prediction
0,p20_s0_v0,Tvende floder flyder om helvedes hegn.,NaN,None
1,p20_s0_v1,Den ene hvirvler en sydende strom,den,None
2,p20_s0_v2,med glans af smeltet metal.,NaN,None
3,p20_s0_v3,"Den anden vælter sorte,",den,None
4,p20_s0_v4,iskolde vande.,NaN,None


In [116]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (
    (anaphora_testset.anaphora_across == "x")
    & (anaphora_testset.anaphora_gold == anaphora_testset.anaphora_across_prediction)
).sum()

false_positive = (
    anaphora_testset.anaphora_across_prediction.notna()
    & (anaphora_testset.anaphora_gold != anaphora_testset.anaphora_across_prediction)
).sum()

true_negative = (anaphora_testset.anaphora_gold.isna() & anaphora_testset.anaphora_across_prediction.isna()).sum()

false_negative = (
    (anaphora_testset.anaphora_across == "x") & (anaphora_testset.anaphora_across_prediction.isna())
).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")

print(f"F1 score: {f1_score * 100:2f}%")

Precision: 9.302326%
Recall: 11.764706%
F1 score: 10.389610%


In [119]:
## Eyeballing errors
condition = (anaphora_testset.anaphora_across == "x") & (
    anaphora_testset.anaphora_gold == anaphora_testset.anaphora_across_prediction
)

anaphora_testset[condition]

,filename,book,verse_id,language,text,anaphora_gold,anaphora_stanza,anaphora_line,anaphora_across,anaphora_prediction,anaphora_line_prediction,anaphora_stanza_prediction,anaphora_across_prediction
220,257_Skolesang_no-nb_digibok_2016051048054,2016051048054,p257_s4_v0,rm,"Ja, der en herlig skat jeg faar",ja,NaN,NaN,x,NaN,None,None,ja
1344,1831_VII_4_no-nb_digibok_2021041348661,2021041348661,p1831_s4_v0,rm,"Min frelser, o vilde du kommel!",min frelser,NaN,NaN,x,NaN,None,None,min frelser
2614,2885_As_leva_no-nb_digibok_2014073108102,2014073108102,p2885_s1_v0,lm,"Aa leva, det er i livet",aa leva det er,x,NaN,x,NaN,None,None,aa leva det er
2618,2885_As_leva_no-nb_digibok_2014073108102,2014073108102,p2885_s2_v0,lm,"Aa leva, det er aa gløyma",aa leva det er,x,NaN,x,NaN,aa,None,aa leva det er


## Alliteration

Evaluate precision / recall on the following alliteration types: 
- word-initial letter (with text)
- word-initial sound (with transcriptions)
- word-initial letter cluster (hv / sj / kj)

In [179]:
from poetry_analysis.alliteration import find_line_alliterations

alllit_testset = pd.read_excel("evaluation_data/norn_dikt_testsett_allitterasjon.xlsx")

# Reformat the gold reference to lists of lists
# alllit_testset["alliteration_gold"] = alllit_testset.alliteration_gold.apply(lambda allit: [words.split() for words in allit.split(";")] if isinstance(allit, str) else None)
alllit_testset["alliteration_gold"] = alllit_testset.alliteration_gold.str.strip()

# Extract words with alliterating and reformat to match the gold standard
alllit_testset["alliteration_prediction"] = alllit_testset.text.apply(find_line_alliterations).apply(
    lambda l: "; ".join(" ".join(words) for words in l) if l else None
)

In [180]:
# EVALUATE
# Count true positives, true negatives, false positives and false negatives
true_positive = (alllit_testset.alliteration_gold.notna() & alllit_testset.alliteration_prediction.notna()).sum()

false_positive = (alllit_testset.alliteration_gold.isna() & alllit_testset.alliteration_prediction.notna()).sum()

true_negative = (alllit_testset.alliteration_gold.isna() & alllit_testset.alliteration_prediction.isna()).sum()

false_negative = (alllit_testset.alliteration_gold.notna() & alllit_testset.alliteration_prediction.isna()).sum()

# Calculate accuracy metrics
precision = true_positive / (true_positive + false_positive)

recall = true_positive / (true_positive + false_negative)

f1_score = (2 * true_positive) / (2 * true_positive + false_positive + false_negative)

print(f"Precision: {precision * 100:2f}%")
print(f"Recall: {recall * 100:2f}%")
print(f"F1 score: {f1_score * 100:2f}%")

Precision: 42.655699%
Recall: 90.298507%
F1 score: 57.940942%


In [176]:
condition = alllit_testset.alliteration_gold.isna() & alllit_testset.alliteration_prediction.notna()
alllit_testset[condition][["verse_id", "alliteration_prediction"]].to_csv("false_positive_alliterations.csv")

In [171]:
alllit_testset.alliteration_prediction

0       floder flyder; helvedes hegn
1                      sydende strom
2                               None
3                               None
4                               None
                    ...             
3100                         dyb din
3101                            None
3102                        møde mit
3103               hvilken herlighed
3104                          da der
Name: alliteration_prediction, Length: 3105, dtype: object